# 5.1 · 客户端特性：征询、根目录与采样

**Client features: elicitation, roots and sampling**

> **学完本课你能：** 解释征询、根目录和采样分别让服务器向客户端索取什么，说明它们如今如何承载于 InputRequiredResult 之中，并把已弃用的根目录和采样迁移为工具参数和直接调用 LLM。

官方阅读材料：[Understanding MCP clients](https://modelcontextprotocol.io/docs/2026-07-28/learn/client-concepts) · [Spec: Roots](https://modelcontextprotocol.io/specification/2026-07-28/client/roots) · [Spec: Sampling](https://modelcontextprotocol.io/specification/2026-07-28/client/sampling)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 准备：客户端声明什么取决于它的回调
在 Python SDK 中，你通过<strong>传入回调</strong>来声明一项客户端特性（Client Features）。`Client(elicitation_callback=...)` 会声明 `elicitation: {form, url}`，`list_roots_callback=` 会声明 `roots`，`sampling_callback=` 会声明 `sampling`。服务器会在请求的 `_meta.io.modelcontextprotocol/clientCapabilities` 中看到这些能力。我们用一个回显所收到内容的工具来验证这一点。

In [ ]:
import json
import logging
import warnings

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

logging.getLogger("mcp").setLevel(logging.CRITICAL)    # keep the SDK's own server-side logs quiet
warnings.filterwarnings("ignore", message=".*deprecated.*")


def wire(model) -> str:
    """A pydantic MCP object exactly as it travels on the wire."""
    return json.dumps(model.model_dump(by_alias=True, exclude_none=True, mode="json"), indent=2)


probe = MCPServer("capability-probe")


@probe.tool()
def my_capabilities(ctx: Context) -> str:
    """Return the client capabilities declared on this request."""
    caps = ctx.client_capabilities
    return json.dumps(caps.model_dump(by_alias=True, exclude_none=True) if caps else {})


async def noop(*args):          # placeholder callbacks: we only care about what gets declared
    return types.ErrorData(code=-1, message="unused")

async with Client(probe) as plain:
    print("no callbacks      ->", (await plain.call_tool("my_capabilities", {})).content[0].text)

async with Client(probe, elicitation_callback=noop, list_roots_callback=noop, sampling_callback=noop) as rich:
    print("all three         ->", (await rich.call_tool("my_capabilities", {})).content[0].text)

### 2. 征询：旅行服务器询问预订详情
这是文档中的巴塞罗那示例。在 `MCPServer` 中，2026 年惯用的提问方式是使用<strong>解析器（Resolver）</strong>：一个返回 `Elicit(message, Schema)` 的函数。你通过 `Annotated[..., Resolve(fn)]` 把它挂到某个工具参数上。框架会把这个问题转换成 `InputRequiredResult`，当客户端带着答案重试时，框架会注入结果并运行工具函数体。把参数标注为 `ElicitationResult[Schema]`，你就能针对全部三种结果进行分支处理。

这个 Pydantic 模型会被渲染为<strong>受限的扁平模式（Schema）</strong>：一个布尔值、一个字符串枚举（`Literal`）和一个带默认值的布尔值。嵌套模型会被拒绝。

In [ ]:
from typing import Annotated, Literal

from pydantic import BaseModel, Field

from mcp.server.elicitation import AcceptedElicitation, ElicitationResult
from mcp.server.mcpserver import Elicit, Resolve


class BookingDetails(BaseModel):
    confirmBooking: bool = Field(description="Confirm the booking (Flights + Hotel = $3,000)")
    seatPreference: Literal["window", "aisle", "no preference"] = Field(description="Preferred seat type")
    travelInsurance: bool = Field(default=False, description="Add travel insurance ($150)")


travel = MCPServer("travel")


def ask_booking_details(destination: str) -> Elicit[BookingDetails]:
    return Elicit(f"Please confirm your {destination} vacation booking details:", BookingDetails)


@travel.tool()
def book_trip(destination: str,
              details: Annotated[ElicitationResult[BookingDetails], Resolve(ask_booking_details)]) -> str:
    """Book a vacation package after the user confirms the details."""
    if isinstance(details, AcceptedElicitation):
        d = details.data
        if not d.confirmBooking:
            return "User reviewed the form but did not confirm. Nothing booked."
        extra = " + insurance" if d.travelInsurance else ""
        return f"Booked {destination}: {d.seatPreference} seat{extra}."
    return f"Booking not made: user chose '{details.action}'."


async with Client(travel) as c:
    tool = (await c.list_tools()).tools[0]
    print("book_trip input schema (what the LLM fills):", json.dumps(tool.input_schema))

接下来是<strong>客户端这一侧</strong>。真实的宿主（Host）会渲染一个表单。我们脚本化的“用户”会打印出界面上应当显示的内容（哪个服务器在提问、消息内容、各个字段），然后作答。我们运行三次，分别观察 `accept`、`decline` 和 `cancel`。

In [ ]:
script = []          # the scripted user's next answers


async def form_ui(context, params: types.ElicitRequestFormParams):
    """Stand-in for a host's elicitation dialog."""
    print(f"  ┌ [travel server] asks: {params.message}")
    for name, prop in params.requested_schema["properties"].items():
        choices = prop.get("enum")
        print(f"  │  {name:<16} {prop['type']:<8} {choices or ''} default={prop.get('default', '-')}")
    action, content = script.pop(0)
    print(f"  └ user -> {action} {content or ''}")
    return types.ElicitResult(action=action, content=content)


script[:] = [("accept", {"confirmBooking": True, "seatPreference": "window", "travelInsurance": True}),
             ("decline", None),
             ("cancel", None)]

async with Client(travel, elicitation_callback=form_ui) as client:
    for _ in range(3):
        result = await client.call_tool("book_trip", {"destination": "Barcelona"})
        print("tool result:", result.content[0].text, "\n")

### 3. 深入内部：线路上的输入请求
`client.call_tool` 隐藏了这次往返。`client.session.call_tool(..., allow_input_required=True)` 则会返回原始的中间结果，这里就是 `InputRequiredResult`，其中<em>包含</em>了 `elicitation/create` 请求。第 5.3 课会手动驱动这个循环。现在先看看它的结构。

In [ ]:
async with Client(travel, elicitation_callback=form_ui) as client:
    interim = await client.session.call_tool("book_trip", {"destination": "Barcelona"}, allow_input_required=True)
    print(type(interim).__name__)
    text = wire(interim)
    print(text[:1400] + ("\n  ..." if len(text) > 1400 else ""))

注意 `resultType: "input_required"`、`inputRequests` 中由服务器选定的键，以及被 SDK <strong>加密</strong>过的 `requestState`（请求状态）。客户端必须原封不动地把它回传。

### 4. 制造故障：没有能力，就没有提问
服务器<strong>不得（MUST NOT）</strong>发送客户端未声明的输入请求。<em>不</em>提供 `elicitation_callback` 进行连接，SDK 就会以规范中的 `-32021` MissingRequiredClientCapability（缺少必需客户端能力）错误拒绝，并指明缺少的是什么。

In [ ]:
from mcp.shared.exceptions import MCPError

async with Client(travel) as client:          # no elicitation_callback -> no elicitation capability
    try:
        await client.call_tool("book_trip", {"destination": "Barcelona"})
    except MCPError as err:
        print("code:", err.error.code)
        print("message:", err.error.message)
        print("data:", err.error.data)

### 5. 采样（已弃用）：服务器借用宿主的模型
在弃用窗口期内，采样（Sampling）仍然可用。解析器返回一个 `Sample(...)`，其中包含消息、必填的 `max_tokens` 和 `model_preferences`。客户端的 `sampling_callback` 是宿主展示提示词供审批、根据提示和优先级<strong>挑选模型</strong>、调用其 LLM 并展示答案供审核的地方。我们的模拟宿主有三个模型，并按规范所描述的方式进行提示匹配（按顺序进行子字符串匹配，然后看优先级）。

In [ ]:
from mcp.server.mcpserver import Sample

flights = MCPServer("flights")
FLIGHTS = "IB3101 07:10 €180 direct | VY1307 06:30 €95 1 stop | BA478 21:40 €70 red-eye"


def analyse(preferences: str) -> Sample:
    return Sample(
        [types.SamplingMessage(role="user", content=types.TextContent(
            type="text", text=f"Recommend the best flight.\n{FLIGHTS}\nUser preferences: {preferences}"))],
        max_tokens=200,
        system_prompt="You are a travel expert.",
        model_preferences=types.ModelPreferences(
            hints=[types.ModelHint(name="sonnet"), types.ModelHint(name="claude")],
            cost_priority=0.3, speed_priority=0.2, intelligence_priority=0.9),
    )


@flights.tool()
def find_best_flight(preferences: str,
                     advice: Annotated[types.CreateMessageResult, Resolve(analyse)]) -> str:
    """Recommend a flight using the client's LLM (deprecated sampling)."""
    return f"[{advice.model}] {advice.content.text}"


HOST_MODELS = {"claude-opus-5-5": {"intelligence": 0.95, "speed": 0.4, "cost": 0.2},
               "claude-haiku-fast": {"intelligence": 0.6, "speed": 0.95, "cost": 0.9},
               "gemini-pro": {"intelligence": 0.9, "speed": 0.6, "cost": 0.5}}


def choose_model(prefs: types.ModelPreferences | None) -> str:
    for hint in (prefs.hints or []) if prefs else []:
        matches = [m for m in HOST_MODELS if hint.name and hint.name in m]
        if matches:
            return matches[0]
    p = prefs or types.ModelPreferences()
    score = lambda m: ((p.intelligence_priority or 0) * HOST_MODELS[m]["intelligence"] +
                       (p.speed_priority or 0) * HOST_MODELS[m]["speed"] +
                       (p.cost_priority or 0) * HOST_MODELS[m]["cost"])
    return max(HOST_MODELS, key=score)


async def host_llm(context, params: types.CreateMessageRequestParams):
    model = choose_model(params.model_preferences)
    print(f"  [host] approve sampling? prompt={params.messages[0].content.text.splitlines()[0]!r} "
          f"maxTokens={params.max_tokens} -> approved, using {model}")
    reply = "VY1307: morning departure, one stop, cheapest option that fits."   # a deterministic 'LLM'
    return types.CreateMessageResult(role="assistant", model=model, stop_reason="endTurn",
                                     content=types.TextContent(type="text", text=reply[: params.max_tokens * 4]))


async with Client(flights, sampling_callback=host_llm) as client:
    interim = await client.session.call_tool("find_best_flight", {"preferences": "morning, max 1 stop"},
                                              allow_input_required=True)
    request = next(iter(interim.input_requests.values()))
    print("input request on the wire:\n", wire(request)[:900])
    final = await client.call_tool("find_best_flight", {"preferences": "morning, max 1 stop"})
    print("\nfinal:", final.content[0].text)

我们的宿主里没有 “sonnet” 模型，所以第一个提示没有命中，而 `"claude"` 匹配成功。这正是规范所描述的那种建议性的、基于子字符串的行为。

### 6. 同一个服务器，旧协议：旧版模式使用服务器发起的请求
使用 `mode="legacy"` 时，客户端讲的是 2025-11-25 协议，SDK 会在调用中途把 `sampling/createMessage` 作为一个真正的<strong>服务器到客户端请求</strong>发送出去，这就是 MRTR 之前的做法。解析器代码不需要改动。不过，像 `ctx.elicit(...)` 这样旧式的命令式辅助函数只能在旧时代协议（Legacy Era）下工作。在 2026-07-28 连接上没有反向通道，它们会失败。

In [ ]:
legacy_demo = MCPServer("legacy-demo")


class Answer(BaseModel):
    name: str


@legacy_demo.tool()
async def ask_name_old_style(ctx: Context) -> str:
    """Ask for a name the pre-2026 way (server-initiated elicitation/create)."""
    result = await ctx.elicit("What is your name?", Answer)
    return f"{result.action}: {getattr(result, 'data', None)}"


async def name_ui(context, params):
    return types.ElicitResult(action="accept", content={"name": "Ada"})

async with Client(flights, mode="legacy", sampling_callback=host_llm) as old:
    print("legacy era", old.protocol_version, "->", (await old.call_tool(
        "find_best_flight", {"preferences": "cheap"})).content[0].text)

for mode in ["legacy", "auto"]:
    async with Client(legacy_demo, mode=mode, elicitation_callback=name_ui) as c:
        try:
            r = await c.call_tool("ask_name_old_style", {})
            print(f"{c.protocol_version}: ctx.elicit ->", r.content[0].text)
        except Exception as err:
            print(f"{c.protocol_version}: ctx.elicit -> {type(err).__name__}: {err}")

这就是用代码讲述的迁移课：<strong>用解析器或显式的 `InputRequiredResult`（需输入结果）编写新服务器</strong>（第 5.3 课），而不是使用服务器发起的调用。

### 7. 带工具的采样（同样已弃用）：需要 `sampling.tools`
服务器可以为宿主的模型提供一组仅限于该请求的工具。如果没有 `sampling.tools` 能力（Capability），SDK 会拒绝发送这样的请求。有了这项能力，回调会收到 `tools` 和 `toolChoice`，模型可以用 `tool_use` 块和 `stopReason: "toolUse"` 作答。

In [ ]:
weather = MCPServer("weather")
GET_WEATHER = types.Tool(name="get_weather", description="Get current weather for a city",
                         input_schema={"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]})


def plan(question: str) -> Sample:
    return Sample([types.SamplingMessage(role="user", content=types.TextContent(type="text", text=question))],
                  max_tokens=300, tools=[GET_WEATHER], tool_choice=types.ToolChoice(mode="auto"))


@weather.tool()
def compare_weather(question: str,
                    step: Annotated[types.CreateMessageResultWithTools, Resolve(plan)]) -> str:
    """Let the client's model decide which weather lookups to make."""
    return json.dumps(step.model_dump(by_alias=True, exclude_none=True, mode="json"))


async def tool_llm(context, params):
    print("  [host] tools offered:", [t.name for t in params.tools or []], "| toolChoice:", params.tool_choice.mode)
    return types.CreateMessageResultWithTools(
        role="assistant", model="claude-opus-5-5", stop_reason="toolUse",
        content=[types.ToolUseContent(type="tool_use", id="call_1", name="get_weather", input={"city": "Paris"}),
                 types.ToolUseContent(type="tool_use", id="call_2", name="get_weather", input={"city": "London"})])

async with Client(weather, sampling_callback=tool_llm) as c:          # sampling, but not sampling.tools
    try:
        await c.call_tool("compare_weather", {"question": "Paris or London?"})
    except MCPError as err:
        print("without sampling.tools ->", err.error.code, err.error.message)

tools_cap = types.SamplingCapability(tools=types.SamplingToolsCapability())
async with Client(weather, sampling_callback=tool_llm, sampling_capabilities=tools_cap) as c:
    out = await c.call_tool("compare_weather", {"question": "Paris or London?"})
    print("with sampling.tools    ->", out.content[0].text)

在完整的循环中，服务器接下来会运行两次 `get_weather`，并发送一个<em>新的</em>采样请求，其历史记录以一条<strong>只</strong>包含 `tool_result` 块（`toolUseId` 为 `call_1` 和 `call_2`）的用户消息结尾，如此反复，直到 `endTurn`。

### 8. 根目录（已弃用），以及推荐的替代方案
先看已弃用的方式。一个 `ListRoots()` 解析器让服务器向客户端索取其 `file://` 根目录（Roots）。然后是<strong>迁移方案</strong>：服务器在启动时从<strong>配置</strong>中获取允许的目录，工具则把目录作为<strong>参数</strong>接收。它通过解析每一条路径（这能挫败 `..` 和符号链接）并检查其是否仍位于某个允许的目录内，来校验每一条路径。

In [ ]:
import tempfile
from pathlib import Path

from mcp.server.mcpserver import ListRoots
from mcp.server.mcpserver.exceptions import ToolError

workspace = Path(tempfile.mkdtemp())
(workspace / "itineraries").mkdir()
(workspace / "itineraries" / "barcelona.md").write_text("Day 1: Sagrada Familia")
(workspace / "secrets.txt").write_text("do not leak")

# --- deprecated: roots via an input request ---
roots_server = MCPServer("roots-demo")


def fetch_roots() -> ListRoots:
    return ListRoots()


@roots_server.tool()
def list_workspace(roots: Annotated[types.ListRootsResult, Resolve(fetch_roots)]) -> list[str]:
    """List files under the client's roots (deprecated mechanism)."""
    files = []
    for root in roots.roots:
        base = Path(str(root.uri).removeprefix("file://"))
        files += [str(p.relative_to(base)) for p in sorted(base.rglob("*")) if p.is_file()]
    return files


async def my_roots(context):
    return types.ListRootsResult(roots=[types.Root(uri=f"file://{workspace / 'itineraries'}", name="Itineraries")])

async with Client(roots_server, list_roots_callback=my_roots) as c:
    print("roots-based listing:", (await c.call_tool("list_workspace", {})).structured_content)

# --- recommended: server configuration + explicit tool parameter ---
ALLOWED_DIRS = [(workspace / "itineraries").resolve()]       # e.g. from argv/env when the host launches the server
files_server = MCPServer("files")


@files_server.tool()
def read_text_file(directory: str, filename: str) -> str:
    """Read a text file from a directory the server is configured to allow."""
    target = (Path(directory) / filename).resolve()
    if not any(target.is_relative_to(allowed) for allowed in ALLOWED_DIRS):
        raise ToolError(f"{filename!r} resolves outside the allowed directories")
    return target.read_text()


async with Client(files_server) as c:
    ok = await c.call_tool("read_text_file", {"directory": str(workspace / "itineraries"), "filename": "barcelona.md"})
    print("allowed  :", ok.is_error, ok.content[0].text)
    bad = await c.call_tool("read_text_file", {"directory": str(workspace / "itineraries"), "filename": "../secrets.txt"})
    print("traversal:", bad.is_error, bad.content[0].text)

路径穿越（Directory Traversal）的尝试在<em>服务器内部</em>失败，并以 `isError: true` 的形式返回，附带一个模型能读懂的原因。我们是有意抛出 `ToolError` 的：`MCPServer` 会隐藏任何<em>意外</em>异常的消息（普通的 `ValueError` 到达客户端时只会显示 “Error executing tool read_text_file”）。另外请注意，目录现在出现在了<strong>工具调用</strong>本身之中，因此宿主的审批界面会向用户准确展示模型想要访问哪个文件夹。

### 9. 可选：把采样迁移为直接调用提供商
采样（Sampling）的替代方案就是普通代码：服务器用<em>自己的</em>密钥调用 LLM API。当设置了 `ANTHROPIC_API_KEY` 时，这个工具会直接询问 Claude。否则它会使用一个确定性的回退结果，以保证笔记本仍能运行。

In [ ]:
import os

direct = MCPServer("flights-direct")


@direct.tool()
def recommend_flight(preferences: str) -> str:
    """Recommend a flight by calling the LLM provider directly (no sampling)."""
    prompt = f"Recommend the best flight in one sentence.\n{FLIGHTS}\nUser preferences: {preferences}"
    if os.environ.get("ANTHROPIC_API_KEY"):
        import anthropic
        response = anthropic.Anthropic().messages.create(
            model="claude-opus-5-5", max_tokens=16000, messages=[{"role": "user", "content": prompt}])
        return "".join(b.text for b in response.content if b.type == "text")
    return "(offline fallback) VY1307 — morning, one stop, cheapest that fits."


async with Client(direct) as c:   # note: no sampling_callback, no sampling capability needed
    print((await c.call_tool("recommend_flight", {"preferences": "morning, max 1 stop"})).content[0].text)

### 动手练习
1. 给 `BookingDetails` 添加一个带 `format: "email"` 的 `email: str` 字段（使用 `pydantic.EmailStr` 或 `Field(json_schema_extra={"format": "email"})`），并检查渲染出的模式（Schema）。然后尝试一个嵌套模型字段，阅读 SDK 抛出的 `TypeError`。规范为什么禁止嵌套？
2. 修改 `host_llm`，让“用户”通过返回 `types.ErrorData(code=-1, message="User denied")` 来<strong>拒绝</strong>采样请求。`client.call_tool` 会抛出什么？当采样被拒绝时，服务器应当怎么做？
3. 在第 8 节中，把 `ToolError` 改为 `ValueError`，比较客户端收到的错误文本。然后在 `itineraries` 中添加一个指向 `secrets.txt` 的符号链接，验证 `resolve()` 仍然能阻止它。
4. 设计题：旅行服务器需要用户的常旅客<strong>密码</strong>来使用里程。请草拟你会发送的输入请求（模式、字段），并解释为什么不允许使用表单模式（Form Mode）。